In [3]:
# ============================================================
# install
# ============================================================
print(" Installing...")
import subprocess, sys
for pkg in ['mtcnn', 'opencv-python']:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print(" Done!\n")

# ============================================================
# Mount
# ============================================================
print(" Mounting Drive...")
from google.colab import drive
drive.mount('/content/drive')
print(" Mounted!\n")

# ============================================================
# Imports
# ============================================================
import os, cv2, numpy as np
from mtcnn import MTCNN
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input

print("="*80)
print("  System")
print("="*80)
print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {len(tf.config.list_physical_devices('GPU')) > 0}")
print("="*80 + "\n")

# ============================================================
# Config
# ============================================================
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

RAW_DIR = "/content/drive/MyDrive/kaggle-genki4k"
CROPPED_DIR = "/content/cropped_faces"

print("="*80)
print(" Smile Detection")
print("="*80)
print(f" Raw: {RAW_DIR}")
print(f" Processed: {CROPPED_DIR}")
print("="*80 + "\n")

# ============================================================
# Face Processing
# ============================================================

def align_face(img, left_eye, right_eye):
    lx, ly = int(left_eye[0]), int(left_eye[1])
    rx, ry = int(right_eye[0]), int(right_eye[1])
    center = ((lx+rx)//2, (ly+ry)//2)
    angle = np.degrees(np.arctan2(ry-ly, rx-lx))
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    return cv2.warpAffine(img, M, (img.shape[1], img.shape[0]))

def detect_crop(detector, img_rgb):
    dets = detector.detect_faces(img_rgb)
    if not dets: return None

    det = max(dets, key=lambda x: x['box'][2]*x['box'][3])
    x, y, w, h = [int(v) for v in det['box']]

    kp = det.get('keypoints', {})
    if 'left_eye' in kp and 'right_eye' in kp:
        try:
            img_rgb = align_face(img_rgb, kp['left_eye'], kp['right_eye'])
        except: pass

    pad_w, pad_h = int(w*0.18), int(h*0.25)
    x1 = max(0, x-pad_w)
    y1 = max(0, y-pad_h)
    x2 = min(img_rgb.shape[1], x+w+pad_w)
    y2 = min(img_rgb.shape[0], y+h+pad_h)

    if x2 <= x1 or y2 <= y1: return None
    face = img_rgb[y1:y2, x1:x2]
    return face if face.size > 0 else None

def process_dataset():
    smile_dir = os.path.join(RAW_DIR, "smile")
    nonsmile_dir = os.path.join(RAW_DIR, "non_smile")

    if not os.path.exists(smile_dir): return False

    os.makedirs(os.path.join(CROPPED_DIR, "smile"), exist_ok=True)
    os.makedirs(os.path.join(CROPPED_DIR, "non_smile"), exist_ok=True)

    print(" Init MTCNN...")
    detector = MTCNN()

    for folder, label in [(smile_dir, "smile"), (nonsmile_dir, "non_smile")]:
        files = [f for f in os.listdir(folder) if f.lower().endswith(('.jpg','.jpeg','.png'))]
        out = os.path.join(CROPPED_DIR, label)
        saved = 0

        print(f"\n{'='*60}")
        print(f"Processing {label.upper()}")
        print(f"{'='*60}")

        for f in tqdm(files):
            img = cv2.imread(os.path.join(folder, f))
            if img is None: continue

            face = detect_crop(detector, cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            if face is not None:
                cv2.imwrite(os.path.join(out, f), cv2.cvtColor(face, cv2.COLOR_RGB2BGR))
                saved += 1

        print(f" Saved: {saved}/{len(files)}")

    return True

# ============================================================
# Training
# ============================================================

# Augmentation - STRONGER
aug = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomContrast(0.3),
    layers.RandomBrightness(0.3),
])

def load_prep(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    return tf.cast(img, tf.float32), tf.cast(label, tf.float32)

def train_prep(path, label):
    img, label = load_prep(path, label)
    img = aug(img)
    img = preprocess_input(img)
    return img, label

def test_prep(path, label):
    img, label = load_prep(path, label)
    img = preprocess_input(img)
    return img, label

def make_ds(paths, labels, training=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels.astype(np.float32)))

    if training:
        ds = ds.shuffle(len(paths), seed=SEED)
        ds = ds.map(train_prep, num_parallel_calls=tf.data.AUTOTUNE)
    else:
        ds = ds.map(test_prep, num_parallel_calls=tf.data.AUTOTUNE)

    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

def tta_predict(model, ds, n=5):
    print("\n TTA...")
    preds = []
    for i in range(n):
        print(f"  {i+1}/{n}...")
        preds.append(model.predict(ds, verbose=0))
    return np.mean(preds, axis=0)

def train_model():
    smile_dir = os.path.join(CROPPED_DIR, "smile")
    nonsmile_dir = os.path.join(CROPPED_DIR, "non_smile")

    if not os.path.exists(smile_dir):
        print(" Dataset not found!")
        return None, None

    # Load
    smile_files = tf.io.gfile.glob(os.path.join(smile_dir, "*"))
    nonsmile_files = tf.io.gfile.glob(os.path.join(nonsmile_dir, "*"))

    all_paths = np.array(smile_files + nonsmile_files)
    all_labels = np.array([1]*len(smile_files) + [0]*len(nonsmile_files))

    # Shuffle
    idx = np.arange(len(all_paths))
    np.random.seed(SEED)
    np.random.shuffle(idx)
    all_paths = all_paths[idx]
    all_labels = all_labels[idx]

    # Split
    n = len(all_paths)
    train_n = int(0.8 * n)
    val_n = int(0.1 * n)

    train_paths, train_labels = all_paths[:train_n], all_labels[:train_n]
    val_paths, val_labels = all_paths[train_n:train_n+val_n], all_labels[train_n:train_n+val_n]
    test_paths, test_labels = all_paths[train_n+val_n:], all_labels[train_n+val_n:]

    print(f"\n{'='*60}")
    print(" Dataset")
    print(f"{'='*60}")
    print(f"Total: {n}, Train: {train_n}, Val: {val_n}, Test: {n-train_n-val_n}")
    print(f"{'='*60}")

    # Datasets
    train_ds = make_ds(train_paths, train_labels, training=True)
    val_ds = make_ds(val_paths, val_labels)
    test_ds = make_ds(test_paths, test_labels)

    # Model
    print("\n" + "="*80)
    print("  Building Model")
    print("="*80)

    base = MobileNetV3Large(include_top=False, weights="imagenet", input_shape=(224,224,3))
    base.trainable = False

    inputs = layers.Input(shape=(224,224,3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)  # Increased
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.2)(x)  # Increased
    outputs = layers.Dense(1, activation="sigmoid")(x)

    model = models.Model(inputs, outputs)

    # Compile - SIMPLE!
    model.compile(
        optimizer='adam',  # Simple
        loss='binary_crossentropy',  # Simple
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )

    model.summary()

    save_path = "/content/drive/MyDrive/best_smile_improved.keras"


    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(save_path, monitor='val_auc', mode='max', save_best_only=True, verbose=1),
        tf.keras.callbacks.EarlyStopping(monitor='val_auc', patience=10, restore_best_weights=True, mode='max', verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_auc', factor=0.5, patience=3, min_lr=1e-7, mode='max', verbose=1)
    ]

    # PHASE 1: Head (5 epochs)
    print("\n" + "="*80)
    print(" PHASE 1: Head (5 epochs)")
    print("="*80)

    h1 = model.fit(train_ds, validation_data=val_ds, epochs=5, callbacks=callbacks, verbose=1)

    # PHASE 2: 20 layers (15 epochs)
    print("\n" + "="*80)
    print(" PHASE 2: 20 layers (15 epochs)")
    print("="*80)

    base.trainable = True
    for layer in base.layers[:-20]:
        layer.trainable = False

    model.compile(optimizer=tf.keras.optimizers.Adam(5e-5), loss='binary_crossentropy',
                  metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])

    h2 = model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=callbacks, verbose=1)

    # PHASE 3: 30 layers (10 epochs)
    print("\n" + "="*80)
    print(" PHASE 3: 30 layers (10 epochs)")
    print("="*80)

    for layer in base.layers[:-30]:
        layer.trainable = False
    for layer in base.layers[-30:]:
        layer.trainable = True

    model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='binary_crossentropy',
                  metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])

    h3 = model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=callbacks, verbose=1)

    print("\n Done!")

    # Eval
    print(f"\n{'='*60}")
    print(" Test")
    print(f"{'='*60}\n")

    print("Without TTA:")
    _, test_acc, test_auc = model.evaluate(test_ds, verbose=1)
    print(f"  Acc: {test_acc:.4f}")
    print(f"  AUC: {test_auc:.4f}")

    print("\nWith TTA:")
    tta_preds = tta_predict(model, test_ds, n=5)
    tta_binary = (tta_preds > 0.5).astype(int).flatten()

    from sklearn.metrics import accuracy_score, roc_auc_score
    tta_acc = accuracy_score(test_labels, tta_binary)
    tta_auc = roc_auc_score(test_labels, tta_preds)

    print(f"  TTA Acc: {tta_acc:.4f}")
    print(f"  TTA AUC: {tta_auc:.4f}")
    print(f"   +{(tta_acc-test_acc)*100:.2f}%")

    print(f"\n Saved: {save_path}")

    return model, {'p1': h1.history, 'p2': h2.history, 'p3': h3.history}

# ============================================================
# Main
# ============================================================

if __name__ == "__main__":
    print("\n" + "="*80)
    print(" STEP 1: Face Detection")
    print("="*80)

    if not os.path.exists(os.path.join(CROPPED_DIR, "smile")):
        print("\n Processing...")
        if not process_dataset():
            print(" Failed!")
            exit(1)
    else:
        print("\n Already done!")

    print("\n" + "="*80)
    print(" STEP 2: Training")
    print("="*80)

    model, history = train_model()

    if model:
        print("\n" + "="*80)
        print("✅ SUCCESS!")
        print("="*80)
        print("\n Model: /content/drive/MyDrive/best_smile_improved.keras")
    else:
        print("\n❌ Failed!")


 Installing...
 Done!

 Mounting Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 Mounted!

  System
TensorFlow: 2.19.0
GPU: True

 Smile Detection
 Raw: /content/drive/MyDrive/kaggle-genki4k
 Processed: /content/cropped_faces


 STEP 1: Face Detection

 Already done!

 STEP 2: Training

 Dataset
Total: 3986, Train: 3188, Val: 398, Test: 400

  Building Model


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MobileNetV3Large (Functional)   │ (None, 7, 7, 960)      │     2,996,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 960)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 960)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │       246,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,242,625 (12.37 MB)

 Trainable params: 246,273 (962.00 KB)

 Non-trainable params: 2,996,352 (11.43 MB)


 PHASE 1: Head (5 epochs)
Epoch 1/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 554ms/step - accuracy: 0.5947 - auc: 0.6140 - loss: 0.7560
Epoch 1: val_auc improved from -inf to 0.78306, saving model to /content/drive/MyDrive/best_smile_improved.keras
100/100 ━━━━━━━━━━━━━━━━━━━━ 81s 672ms/step - accuracy: 0.5950 - auc: 0.6144 - loss: 0.7553 - val_accuracy: 0.7161 - val_auc: 0.7831 - val_loss: 0.5730 - learning_rate: 0.0010
Epoch 2/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 514ms/step - accuracy: 0.6649 - auc: 0.7256 - loss: 0.6131
Epoch 2: val_auc improved from 0.78306 to 0.78330, saving model to /content/drive/MyDrive/best_smile_improved.keras
100/100 ━━━━━━━━━━━━━━━━━━━━ 59s 525ms/step - accuracy: 0.6650 - auc: 0.7256 - loss: 0.6131 - val_accuracy: 0.7261 - val_auc: 0.7833 - val_loss: 0.5710 - learning_rate: 0.0010
Epoch 3/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 507ms/step - accuracy: 0.6888 - auc: 0.7543 - loss: 0.5858
Epoch 3: val_auc improved from 0.78330 to 0.80681, saving model to /content/drive/MyDrive